# AI Lab - Practical Question No. 2
## Randomized Reflex Agent: 1D Grid Simulation

- **Syllabus Area**: Intelligent Agents
- **Difficulty**: 🟢 Beginner
- **Practical Type**: Simulation
- **Core AI Concept**: Randomized Agent Behavior, Handling Unobservable States, Escaping Deterministic State Loops
- **Complexity**: Time Complexity: $O(N)$ average | Space Complexity: $O(N)$

---

### Problem Statement
Construct a Python program to model a **Randomized Reflex Agent** that navigates a 1D grid of 5 rooms.
- If the current room is **dirty** (`1`), the agent cleans it (`Clean` / `Suck`).
- If the room is **clean** (`0`), it selects its next movement (`Left` or `Right`) **randomly** using Python's `random` module to prevent infinite looping in boundary rooms.
- The agent does **not** have access to the global array state (partially observable / unobservable environment).
- The simulation runs until all rooms are clean, outputting the total moves and steps taken.

### Agent PEAS Specification

| PEAS Component | Specification for Question 2 |
| :--- | :--- |
| **P**erformance Measure | Clean all rooms in minimal steps; +10 per clean room, -1 per move/step |
| **E**nvironment | 1D discrete grid of $N=5$ rooms, indexed $[0, 1, 2, 3, 4]$ with physical boundary walls |
| **A**ctuators | `Clean`, `Left` (decrement index, bounded at 0), `Right` (increment index, bounded at $N-1$) |
| **S**ensors | Local dirt sensor only: returns $1$ (Dirty) or $0$ (Clean) for current room |

### Randomized Condition-Action Rules

$$\text{Action} = f(\text{Percept})$$

- **Rule 1 (Dirty Room)**: If $\text{Percept} = 1 \implies \text{Action} = \text{Clean}$
- **Rule 2 (Clean Room)**: If $\text{Percept} = 0 \implies \text{Action} \sim \text{Uniform}(\{\text{Left}, \text{Right}\})$

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path when running from notebooks/
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.randomized_agent import (
    RandomizedReflexAgent,
    OneDimensionalGridEnvironment,
    run_randomized_reflex_simulation,
    ROOM_CLEAN,
    ROOM_DIRTY,
    ACTION_CLEAN,
    ACTION_LEFT,
    ACTION_RIGHT,
)
print("Loaded RandomizedReflexAgent and OneDimensionalGridEnvironment successfully.")

## 1. Primary Simulation: Sample Scenario

- **Initial State Array**: `[1, 0, 0, 1, 1]` (where `1` indicates dirty)
- **Agent Starting Index**: `2` (middle room)
- **Reproducibility Seed**: `58`
- **Expected Output**: *All rooms clean in 12 steps.*

In [ ]:
# Initialize environment and agent
sample_state = [1, 0, 0, 1, 1]
start_idx = 2

env = OneDimensionalGridEnvironment(initial_state=sample_state, start_index=start_idx)
agent = RandomizedReflexAgent(seed=58)

print(f"Initial Grid State:  {env.grid}")
print(f"Agent Start Index:   Room [{env.agent_index}]\n")
print("--- Step-by-Step Simulation Trace ---")

# Execute simulation loop
while not env.is_all_clean():
    record = env.execute_step(agent)
    print(record.format_trace())

print("-" * 60)
print(f"All rooms clean in {env.step_count} steps.")
print(f"Total Navigation Moves : {env.move_count}")
print(f"Total Cleaning Actions : {env.clean_count}")
print(f"Final Grid State       : {env.grid}")

## 2. Theoretical Analysis: Escaping Boundary Room Loops

### Why Deterministic Reflex Agents Fail
If a deterministic reflex agent is in Room 0 (clean) and its hardcoded rule is `if Clean -> Left`, it will repeatedly execute `Left`, bump into the wall, and remain trapped in Room 0 indefinitely.

### How Randomization Solves It
With stochastic choice $P(\text{Right}) = 0.5$, the probability that the agent fails to escape after $k$ steps is $(0.5)^k$.
As $k$ grows, $(0.5)^k \to 0$, guaranteeing escape with probability 1.

In [ ]:
# Demonstrate boundary behavior at Room 0
boundary_env = OneDimensionalGridEnvironment(initial_state=[0, 1, 1, 1, 1], start_index=0)
boundary_agent = RandomizedReflexAgent(seed=42)

print(f"Starting at boundary room: Room [{boundary_env.agent_index}], State: {boundary_env.grid}")
for _ in range(5):
    rec = boundary_env.execute_step(boundary_agent)
    print(rec.format_trace())

## 3. Empirical Analysis: Monte Carlo Benchmark (100 Trials)

Because the agent is stochastic, execution length varies. We conduct a Monte Carlo study across 100 trials to observe the empirical distribution of steps.

In [ ]:
trials = 100
steps_record = []
moves_record = []

for _ in range(trials):
    res = run_randomized_reflex_simulation(
        grid=[1, 0, 0, 1, 1],
        start_index=2,
        seed=None, # Pure stochastic entropy
        max_steps=1000,
    )
    steps_record.append(res.total_steps)
    moves_record.append(res.total_moves)

avg_steps = sum(steps_record) / len(steps_record)
avg_moves = sum(moves_record) / len(moves_record)
min_steps = min(steps_record)
max_steps = max(steps_record)

print("=== Monte Carlo Benchmark Results (100 Trials) ===")
print(f"Mean Total Steps Taken : {avg_steps:.2f}")
print(f"Mean Navigation Moves  : {avg_moves:.2f}")
print(f"Minimum Steps Taken    : {min_steps}")
print(f"Maximum Steps Taken    : {max_steps}")

## 4. Optional Challenge: Step Limit Safety Ceiling

To prevent runaway execution loops in pathological environments, we enforce a strict `max_steps` ceiling.

In [ ]:
# Test safety ceiling with a low max_steps constraint
constrained_env = OneDimensionalGridEnvironment(initial_state=[1, 0, 0, 1, 1], start_index=2)
agent_constrained = RandomizedReflexAgent(seed=12)

result_constrained = constrained_env.simulate(agent=agent_constrained, max_steps=5)
print(result_constrained.summary())

## 5. University Viva Voce Reference Notes

### Q1: Deterministic vs. Stochastic Behavior
> **Answer**: A **deterministic agent** chooses actions using a fixed mapping from percept to action: $f(p) \to a$. Given the same state or percept, its action is always identical. A **stochastic agent** chooses actions based on probability distributions: $P(a \mid p)$. While deterministic agents are predictable and reproducible, they easily become trapped in infinite loops in partially observable environments. Stochastic agents break symmetry and ensure that all reachable states are visited with probability 1.

### Q2: State Loops in Simple Reflex Agents
> **Answer**: Simple reflex agents have **no internal memory** and base their choices solely on the current percept. In an unobservable or partially observable environment where the agent does not perceive its global position (e.g. knowing only that the room is clean, but not which room it is in), a deterministic rule (e.g., `Clean -> Left`) will force the agent into an infinite loop when it hits a boundary wall, endlessly bumping against it.

### Q3: Rationale for Randomization in Unknown Environments
> **Answer**: As demonstrated by Russell & Norvig (AIMA), randomized action selection is the simplest and most elegant mechanism to escape deterministic infinite loops without adding complex state memory. It enables the agent to explore all rooms via a 1D random walk.